# squig.link Update
Updates measurements and results for the supported squig.link sites.

The stock `db.ipynb` imports `Oratory1990Crawler`, which needs a system Ghostscript
installation. squig.link does not, so this notebook contains only the squig.link workflow.

The crawler locates new measurements by combining each site's `phone_book.json` with its
`config.js` (channels and samples) and probing the data directory, because squig.link no
longer serves HTML directory listings.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
from pathlib import Path
ROOT_PATH = Path().resolve().parent
if str(ROOT_PATH) not in sys.path:
    sys.path.insert(ROOT_PATH, 1)


In [ ]:
from IPython.display import display
from autoeq.constants import PEQ_CONFIGS, DEFAULT_BASS_BOOST_GAINS
from autoeq.batch_processing import batch_processing
from dbtools.squig_crawler import SquigCrawlerManager, _squig_rigs
from dbtools.prune_results import prune_results
from dbtools.update_result_indexes import update_all_indexes
from dbtools.constants import TARGETS_PATH, MEASUREMENTS_PATH, RESULTS_PATH

manager = SquigCrawlerManager()
print(f'{len(_squig_rigs)} supported sites')


## Clear phone books
Phone books are cached locally. Deleting them makes the crawler download fresh names.


In [ ]:
for fp in MEASUREMENTS_PATH.glob('**/phone_book*.json'):
    fp.unlink()


## Resolve new measurements
Run the cell of each site below. It crawls the new entries and shows the prompt widget.

* Confirm the name, form and rig of each new measurement. Resolving one measurement of a
  model automatically resolves its other channels and samples.
* When the prompt list is exhausted, run the same cell again: it rebuilds the list with the
  still unresolved measurements. Repeat until no prompts are left.
* If a new manufacturer shows up, add it to `dbtools/manufacturers.tsv` (true name and
  optional aliases separated by tabs) and retry.


In [ ]:
def start(site):
    crawler = manager.crawler(site)
    crawler.crawl()
    return crawler


def next_prompts(crawler, max_prompts=100):
    for item in crawler.crawl_index.items:
        crawler.resolve(item)
    crawler.create_prompts(max_prompts=max_prompts)
    display(crawler.widget)


### Auriculares Argentina

In [ ]:
crawler = start('Auriculares Argentina')
next_prompts(crawler)


### Bakkwatan

In [ ]:
crawler = start('Bakkwatan')
next_prompts(crawler)


### DHRME

In [ ]:
crawler = start('DHRME')
next_prompts(crawler)


### Fahryst

In [ ]:
crawler = start('Fahryst')
next_prompts(crawler)


### Filk

In [ ]:
crawler = start('Filk')
next_prompts(crawler)


### freeryder05

In [ ]:
crawler = start('freeryder05')
next_prompts(crawler)


### Harpo

In [ ]:
crawler = start('Harpo')
next_prompts(crawler)


### Hi End Portable

In [ ]:
crawler = start('Hi End Portable')
next_prompts(crawler)


### Jaytiss

In [ ]:
crawler = start('Jaytiss')
next_prompts(crawler)


### Kazi

In [ ]:
crawler = start('Kazi')
next_prompts(crawler)


### kr0mka

In [ ]:
crawler = start('kr0mka')
next_prompts(crawler)


### Kuulokenurkka

In [ ]:
crawler = start('Kuulokenurkka')
next_prompts(crawler)


### Regan Cipher

In [ ]:
crawler = start('Regan Cipher')
next_prompts(crawler)


### RikudouGoku

In [ ]:
crawler = start('RikudouGoku')
next_prompts(crawler)


### Super Review

In [ ]:
crawler = start('Super Review')
next_prompts(crawler)


### Ted's Squig Hoard

In [ ]:
crawler = start('Ted's Squig Hoard')
next_prompts(crawler)


### ToneDeafMonk

In [ ]:
crawler = start('ToneDeafMonk')
next_prompts(crawler)


## Process measurements
Downloads the resolved measurements and writes averaged frequency responses to `measurements/`.
Only measurements without an existing file are processed.


In [ ]:
manager.process(new_only=True)


## Prune obsolete results
Removes results that no longer have a matching measurement (renames, removed measurements).
Run with `dry_run=True` first.


In [ ]:
prune_results(databases=[
    'Auriculares Argentina',
    'Bakkwatan',
    'DHRME',
    'Fahryst',
    'Filk',
    'freeryder05',
    'Harpo',
    'Hi End Portable',
    'Jaytiss',
    'Kazi',
    'kr0mka',
    'Kuulokenurkka',
    'Regan Cipher',
    'RikudouGoku',
    'Super Review',
    "Ted's Squig Hoard",
    'ToneDeafMonk',
], dry_run=True)


## Generate results
Creates the equalizer settings and plots in `results/` for the new measurements only.


In [ ]:
def update_results(source_db, form, target, rig=None, **override_kwargs):
    input_dir = MEASUREMENTS_PATH.joinpath(source_db, 'data', form)
    if rig is not None:
        input_dir = input_dir.joinpath(rig)
    kwargs = {
        'input_dir': input_dir,
        'output_dir': RESULTS_PATH.joinpath(source_db, f'{rig} {form}' if rig is not None else form),
        'target': TARGETS_PATH.joinpath(target.replace('.csv', '') + '.csv'),
        'bass_boost_gain': DEFAULT_BASS_BOOST_GAINS[target],
        'bass_boost_fc': 105, 'bass_boost_q': 0.7,
        'parametric_eq': True, 'ten_band_eq': True, 'convolution_eq': True,
        'parametric_eq_config': [PEQ_CONFIGS['4_PEAKING_WITH_LOW_SHELF'], PEQ_CONFIGS['4_PEAKING_WITH_HIGH_SHELF']],
        'fs': [44100, 48000],
        'thread_count': 0,
    }
    if override_kwargs:
        kwargs.update(override_kwargs)
    batch_processing(**kwargs)


In [ ]:
update_results('Auriculares Argentina', 'over-ear', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('Auriculares Argentina', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Bakkwatan', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('DHRME', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Fahryst', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Filk', 'over-ear', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('Filk', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('freeryder05', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Harpo', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Hi End Portable', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Jaytiss', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Kazi', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Kazi', 'earbud', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('kr0mka', 'over-ear', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('kr0mka', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('kr0mka', 'earbud', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Kuulokenurkka', 'over-ear', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('Regan Cipher', 'over-ear', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('Regan Cipher', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Regan Cipher', 'earbud', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('RikudouGoku', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Super Review', 'over-ear', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('Super Review', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('Super Review', 'earbud', 'Harman over-ear 2018 without bass', new_only=True)


In [ ]:
update_results('Ted's Squig Hoard', 'in-ear', 'AutoEq in-ear', new_only=True)


In [ ]:
update_results('ToneDeafMonk', 'in-ear', 'AutoEq in-ear', new_only=True)


## Update indexes
Updates recommended results, full results, database specific results, HeSuVi results and the ranking table.


In [ ]:
update_all_indexes()


## Deploy
1. Add files to Git, commit and push
2. Upload webapp data to server
